# 01 · Descarga e inventario del conjunto de datos
**Entrega 1 · cronograma días 1–2.** No necesita GPU.

Qué hace:
1. Conecta Google Drive y entra a la carpeta del proyecto.
2. Descarga el dataset Cassava (iCassava 2019) desde su fuente oficial (≈1,3 GB, solo la primera vez).
3. Construye el **inventario**: una tabla con una fila por imagen.
4. Comprueba que las cantidades coinciden con las publicadas (5.656 / 1.889 / 1.885).

In [1]:
# PREPARACIÓN (igual en todos los notebooks): conecta Drive, entra a la carpeta del proyecto y deja listos los datos.
from google.colab import drive
drive.mount('/content/drive')

PROYECTO = '/content/drive/MyDrive/yuca-comparacion'   # carpeta del proyecto dentro de su Google Drive

import os, sys
os.chdir(PROYECTO)
sys.path.insert(0, PROYECTO)

from src.datos import descargar_dataset
# El .zip se guarda una sola vez en Drive; las fotos se descomprimen en el disco de Colab (más rápido).
RAIZ = str(descargar_dataset(destino='/content/datos', carpeta_zip=PROYECTO + '/data'))
os.environ['YUCA_DATOS'] = RAIZ
print('Proyecto:', os.getcwd())
print('Datos   :', RAIZ)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[1/2] Descargando https://storage.googleapis.com/emcassavadata/cassavaleafdata.zip
      (pesa alrededor de 1,3 GB; puede tardar varios minutos)
            1 MB de 1354 MB
          136 MB de 1354 MB
          272 MB de 1354 MB
          407 MB de 1354 MB
          542 MB de 1354 MB
          677 MB de 1354 MB
          813 MB de 1354 MB
          948 MB de 1354 MB
         1084 MB de 1354 MB
         1219 MB de 1354 MB
         1354 MB de 1354 MB
[2/2] Descomprimiendo...
[ok] Dataset listo en: /content/datos/cassavaleafdata
Proyecto: /content/drive/MyDrive/yuca-comparacion
Datos   : /content/datos/cassavaleafdata


## Inventario
Se abre cada imagen y se anota su partición, clase, resolución, peso y huella MD5 (dos archivos idénticos tienen la misma huella).

In [2]:
from src.datos import construir_inventario
inventario = construir_inventario(RAIZ)
print('Imágenes inventariadas:', len(inventario))
inventario.head()

Imágenes inventariadas: 9430


,particion,clase,archivo,ruta,ancho,alto,modo,bytes,md5
0,train,cbb,train-cbb-0.jpg,train/cbb/train-cbb-0.jpg,666,500,RGB,53498,64428bb9d212a1587e100d9fc6492c90
1,train,cbb,train-cbb-1.jpg,train/cbb/train-cbb-1.jpg,666,500,RGB,57350,bf5f611f3fe6e721f46ca4125ca68133
2,train,cbb,train-cbb-10.jpg,train/cbb/train-cbb-10.jpg,500,500,RGB,134967,81d66d795e1c0b455cacf42ac22e7ef3
3,train,cbb,train-cbb-100.jpg,train/cbb/train-cbb-100.jpg,500,666,RGB,76437,2c7f1139d4a462cbfc84f01ff0bc39d3
4,train,cbb,train-cbb-101.jpg,train/cbb/train-cbb-101.jpg,500,500,RGB,101793,d38097a8dedf44db0f7bc1fe1ca64720


## Distribución por clase y partición

In [3]:
from src.datos import tabla_distribucion, tabla_porcentajes, NOMBRES_CLASES
for sigla, nombre in NOMBRES_CLASES.items():
    print(f'{sigla:8} = {nombre}')
tabla_distribucion(inventario)

cbb      = Tizón bacteriano (Cassava Bacterial Blight)
cbsd     = Estriado marrón (Cassava Brown Streak Disease)
cgm      = Ácaro verde (Cassava Green Mite)
cmd      = Mosaico (Cassava Mosaic Disease)
healthy  = Hoja sana


particion,train,validation,test,total
clase,,,,
cbb,466,156,155,777
cbsd,1443,482,481,2406
cgm,773,258,258,1289
cmd,2658,887,886,4431
healthy,316,106,105,527
TOTAL,5656,1889,1885,9430


In [4]:
tabla_porcentajes(inventario)   # porcentaje de cada clase dentro de cada partición

particion,train,validation,test
clase,,,
cbb,8.2,8.3,8.2
cbsd,25.5,25.5,25.5
cgm,13.7,13.7,13.7
cmd,47.0,47.0,47.0
healthy,5.6,5.6,5.6


## ¿Coincide con lo publicado?

In [5]:
from src.datos import verificar_conteos
coincide = verificar_conteos(inventario)
print('\nTodo coincide' if coincide else '\nHAY DIFERENCIAS: anotarlas en la bitácora antes de continuar')

  train       esperado=5656  encontrado=5656  OK
  validation  esperado=1889  encontrado=1889  OK
  test        esperado=1885  encontrado=1885  OK

Todo coincide


## Resoluciones más frecuentes

In [6]:
inventario.groupby(['ancho', 'alto']).size().sort_values(ascending=False).head(10)

ancho  alto
500    666     2369
888    500     1328
625    500     1073
666    500     1070
500    500      719
       888      512
499    500       46
623    500       18
887    500       16
523    500       14
dtype: int64

## Guardar
`resultados/inventario.csv` contiene los identificadores de las imágenes de cada partición (el profesor pide entregarlos).

In [7]:
os.makedirs('resultados', exist_ok=True)
inventario.to_csv('resultados/inventario.csv', index=False)
tabla_distribucion(inventario).to_csv('resultados/distribucion_clases.csv')
print('Guardado en la carpeta resultados/ del proyecto (Drive).')

Guardado en la carpeta resultados/ del proyecto (Drive).


## Después de ejecutar
- Completar `docs/01_ficha_datos.md` con las cifras reales.
- Anotar en la bitácora: fecha, quién ejecutó, si coincidieron las cantidades y cualquier aviso.
- Subir a GitHub la carpeta `resultados/` y este notebook ya ejecutado.